<a href="https://colab.research.google.com/github/MalakSalehh/Thesis-datasets/blob/main/CLEAN_TRIALtitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!pip install -q groq pandas numpy scipy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.0 MB/s eta 0:00:00


In [21]:
import os
import re
import time
import numpy as np
import pandas as pd

from groq import Groq
from sklearn.metrics import cohen_kappa_score, mean_absolute_error
from scipy.stats import pearsonr

In [20]:
RANDOM_STATE = 42

In [44]:
def get_response(prompt, model="Llama-3.3-70b-versatile", max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are an expert essay grader. "
                            "Follow the rubric strictly and return only the requested format."
                        )
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0.2,
                max_tokens=800
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(5 * (attempt + 1))

    return None

In [7]:

url = "https://raw.githubusercontent.com/MalakSalehh/Thesis-datasets/main/training_set_rel3.tsv"
df = pd.read_csv(url, sep="\t", encoding="latin1")

print(df.shape)
df.head()

(12976, 28)


,essay_id,essay_set,essay,rater1_domain1,rater2_domain1,rater3_domain1,domain1_score,rater1_domain2,rater2_domain2,domain2_score,...,rater2_trait3,rater2_trait4,rater2_trait5,rater2_trait6,rater3_trait1,rater3_trait2,rater3_trait3,rater3_trait4,rater3_trait5,rater3_trait6
0,1,1,"Dear local newspaper, I think effects computer...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",5,4,NaN,9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",4,3,NaN,7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",5,5,NaN,10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,"Dear @LOCATION1, I know having computers has a...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
set1 = df[df["essay_set"] == 1].copy()
set1 = set1[["essay_id", "essay", "domain1_score"]]

set1["essay"] = set1["essay"].astype(str).str.replace("\n", " ", regex=False).str.strip()

print(set1.shape)
set1.head()

(1783, 3)


,essay_id,essay,domain1_score
0,1,"Dear local newspaper, I think effects computer...",8
1,2,"Dear @CAPS1 @CAPS2, I believe that using compu...",9
2,3,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7
3,4,"Dear Local Newspaper, @CAPS1 I have found that...",10
4,5,"Dear @LOCATION1, I know having computers has a...",8


In [9]:

def categorize(score):
    if score <= 5:
        return "Low"
    elif score <= 8:
        return "Medium"
    else:
        return "High"

set1["score_category"] = set1["domain1_score"].apply(categorize)

set1["domain1_score"].value_counts().sort_index()

,count
domain1_score,
2,10
3,1
4,17
5,17
6,110
7,135
8,687
9,334
10,316


In [37]:

cal_low = set1[set1["score_category"] == "Low"].sample(2, random_state=RANDOM_STATE)
cal_med = set1[set1["score_category"] == "Medium"].sample(2, random_state=RANDOM_STATE)
cal_high = set1[set1["score_category"] == "High"].sample(2, random_state=RANDOM_STATE)

calibration = pd.concat([cal_low, cal_med, cal_high]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
calibration


,essay_id,essay,domain1_score,score_category
0,1592,The computers are cool. Do you now I werpsite ...,2,Low
1,995,"I think computers are good, because some peopl...",4,Low
2,686,"Dear local newspaper, I would like to speak up...",9,High
3,1572,"Dear to @CAPS1 this @MONTH1 concern, Computers...",8,Medium
4,1567,"Dear @ORGANIZATION1, @DATE1, everywhere you lo...",11,High
5,146,Dear local newspaper I think that usieng compu...,6,Medium


In [38]:
cal_low = set1[set1["score_category"] == "Low"].sample(2, random_state=RANDOM_STATE)
cal_med = set1[set1["score_category"] == "Medium"].sample(2, random_state=RANDOM_STATE)
cal_high = set1[set1["score_category"] == "High"].sample(2, random_state=RANDOM_STATE)

calibration = pd.concat([cal_low, cal_med, cal_high]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

calibration_ids = calibration["essay_id"].tolist()
pool = set1[~set1["essay_id"].isin(calibration_ids)].copy().reset_index(drop=True)

demo_30 = pool.sample(30, random_state=RANDOM_STATE).reset_index(drop=True)

In [39]:
rubric_text = """
Essay Prompt:
Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.

Rubric:
1 – Undeveloped response with minimal support.
2 – Underdeveloped response with weak reasoning.
3 – Minimally developed argument with limited support.
4 – Adequate argument with some supporting details.
5 – Developed argument with clear reasoning and organization.
6 – Strong persuasive response with well-developed ideas.

Scoring rule:
Two raters assign scores from 1–6.
Final score = sum of both scores.
Final score range: 2–12.
"""

def format_calibration_examples_with_id(calibration_df):
    examples = []
    for i, row in enumerate(calibration_df.itertuples(index=False), start=1):
        examples.append(f"""
Calibration Example {i}

Essay ID: {row.essay_id}

Essay:
{row.essay}

Human Score: {row.domain1_score}
Category: {row.score_category}
""")
    return "\n".join(examples)

calibration_text_with_id = format_calibration_examples_with_id(calibration)

In [40]:
# create both calibration text versions exactly like the notebook styles
calibration_text = format_calibration_examples(calibration)
calibration_text_with_id = format_calibration_examples_with_id(calibration)

In [41]:
def build_full_prompt(essay_text, rubric_text, calibration_text):
    return f"""
You are an expert essay grader evaluating student essays.

{rubric_text}

Calibration Examples:
{calibration_text}

Now evaluate the following essay.

Essay:
{essay_text}

Instructions:
1. Predict the final score between 2 and 12.
2. Predict the category: Low, Medium, or High.
3. Explain your reasoning.

Return your answer in this format:

Final Score:
Category:
Reasoning:
"""

In [42]:
import re

def extract_final_score(text):
    if not isinstance(text, str):
        return None

    match = re.search(r"Final Score:\s*(\d+)", text, re.IGNORECASE)
    if match:
        score = int(match.group(1))
        score = max(2, min(12, score))   # clip to valid range
        return score
    return None

def extract_category(text):
    if not isinstance(text, str):
        return None

    match = re.search(r"Category:\s*(Low|Medium|High)", text, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    return None

In [45]:
results = []

for _, row in demo_30.iterrows():
    essay_id = row["essay_id"]
    essay_text = row["essay"]
    human_score = row["domain1_score"]
    human_category = row["score_category"]

    prompt = build_full_prompt(essay_text, rubric_text, calibration_text)
    output = get_response(prompt)

    predicted_score = extract_final_score(output)
    predicted_category = extract_category(output)

    results.append({
        "essay_id": essay_id,
        "human_score": human_score,
        "human_category": human_category,
        "model_output": output,
        "predicted_score": predicted_score,
        "predicted_category": predicted_category
    })

    print(f"Done essay {essay_id}")
    time.sleep(1)   # optional, just to avoid rate issues

Done essay 66
Done essay 946
Done essay 837
Done essay 1710
Done essay 804
Done essay 1539
Done essay 1042
Done essay 241
Done essay 1211
Done essay 1485
Done essay 941
Done essay 1126
Done essay 1564
Done essay 1730
Done essay 1073
Done essay 922
Done essay 110
Done essay 1588
Done essay 558
Done essay 886
Done essay 355
Done essay 402
Done essay 1519
Done essay 1402
Done essay 325
Done essay 536
Done essay 1351
Done essay 699
Done essay 1691
Done essay 738


In [46]:
results_df = pd.DataFrame(results)
results_df

,essay_id,human_score,human_category,model_output,predicted_score,predicted_category
0,66,9,High,Final Score: 8\nCategory: Medium\nReasoning: T...,8,Medium
1,946,8,Medium,Final Score: 9\nCategory: High\nReasoning: The...,9,High
2,837,8,Medium,Final Score: 8\nCategory: Medium\nReasoning: T...,8,Medium
3,1710,9,High,Final Score: 8\nCategory: Medium\nReasoning: T...,8,Medium
4,804,8,Medium,Final Score: 7\nCategory: Medium\nReasoning: T...,7,Medium
5,1539,9,High,Final Score: 10\nCategory: High\nReasoning: Th...,10,High
6,1042,8,Medium,Final Score: 10\nCategory: High\nReasoning: Th...,10,High
7,241,9,High,Final Score: 7\nCategory: Medium\nReasoning: T...,7,Medium
8,1211,9,High,Final Score: 10\nCategory: High\nReasoning: Th...,10,High
9,1485,9,High,Final Score: 9\nCategory: Medium\nReasoning: T...,9,Medium


In [47]:
print(results_df[[
    "essay_id",
    "human_score",
    "predicted_score",
    "human_category",
    "predicted_category"
]])

print("\nMissing predicted scores:", results_df["predicted_score"].isna().sum())

    essay_id  human_score  predicted_score human_category predicted_category
0         66            9                8           High             Medium
1        946            8                9         Medium               High
2        837            8                8         Medium             Medium
3       1710            9                8           High             Medium
4        804            8                7         Medium             Medium
5       1539            9               10           High               High
6       1042            8               10         Medium               High
7        241            9                7           High             Medium
8       1211            9               10           High               High
9       1485            9                9           High             Medium
10       941           12               11           High               High
11      1126           10                8           High             Medium

In [48]:
clean_df = results_df.dropna(subset=["predicted_score"]).copy()

qwk = cohen_kappa_score(
    clean_df['human_score'],
    clean_df['predicted_score'],
    weights='quadratic',
     labels=list(range(2, 13))
)

pcc, _ = pearsonr(
    clean_df["human_score"],
    clean_df["predicted_score"]
)

mae = mean_absolute_error(
    clean_df["human_score"],
    clean_df["predicted_score"]
)

print("QWK:", qwk)
print("PCC:", pcc)
print("MAE:", mae)

QWK: 0.5590551181102362
PCC: 0.5755390591696988
MAE: 1.1333333333333333


**MTS**

In [54]:
def build_mts_prompt(essay_id, essay_text, rubric_text, calibration_text_with_id):
    return f"""
You are an expert essay grader.

{rubric_text}

Trait guidance:
- Content: relevance of ideas, development of argument, supporting details
- Organization: structure, coherence, logical flow, paragraphing
- Language: grammar, vocabulary, sentence clarity, mechanics

Calibration Examples:
{calibration_text_with_id}

Evaluate the essay using three traits:
1. Content
2. Organization
3. Language

Scoring rules:
- Each trait must be an integer from 1 to 6 only.
- Do not calculate the final score.
- Do not output category.
- Do not output average.
- Base trait judgments strictly on the rubric, trait guidance, and calibration examples.
- Use the exact format below.
- Do not output anything extra.

Essay ID: {essay_id}

Essay:
{essay_text}

Return exactly this format:

Essay ID: {essay_id}
Content Score: <1-6>
Organization Score: <1-6>
Language Score: <1-6>
Reasoning: <2 short sentences>
"""

In [55]:
import numpy as np
import re

def extract_mts_trait_score(text, trait_name):
    if not isinstance(text, str):
        return None
    match = re.search(rf"{trait_name}:\s*(\d+)", text, re.IGNORECASE)
    if match:
        score = int(match.group(1))
        return max(1, min(6, score))
    return None

def compute_mts_score(content, organization, language):
    if None in [content, organization, language]:
        return None, None, None
    avg = np.mean([content, organization, language])
    final_score = int(round(avg * 2))
    final_score = max(2, min(12, final_score))

    if final_score <= 5:
        category = "Low"
    elif final_score <= 8:
        category = "Medium"
    else:
        category = "High"

    return avg, final_score, category


In [56]:
mts_results = []

for _, row in demo_30.iterrows():
    essay_id = row["essay_id"]
    essay_text = row["essay"]
    human_score = row["domain1_score"]
    human_category = row["score_category"]

    prompt = build_mts_prompt(
        essay_id=essay_id,
        essay_text=essay_text,
        rubric_text=rubric_text,
        calibration_text_with_id=calibration_text_with_id
    )

    output = get_response(prompt)

    if output is None:
        print(f"Skipping essay {essay_id} بسبب quota/problem")
        mts_results.append({
            "essay_id": essay_id,
            "human_score": human_score,
            "human_category": human_category,
            "raw_output": None,
            "content_score": None,
            "organization_score": None,
            "language_score": None,
            "average_score": None,
            "predicted_score": None,
            "predicted_category": None
        })
        continue

    content_score = extract_mts_trait_score(output, "Content Score")
    organization_score = extract_mts_trait_score(output, "Organization Score")
    language_score = extract_mts_trait_score(output, "Language Score")

    average_score, predicted_score, predicted_category = compute_mts_score(
        content_score, organization_score, language_score
    )

    mts_results.append({
        "essay_id": essay_id,
        "human_score": human_score,
        "human_category": human_category,
        "raw_output": output,
        "content_score": content_score,
        "organization_score": organization_score,
        "language_score": language_score,
        "average_score": average_score,
        "predicted_score": predicted_score,
        "predicted_category": predicted_category
    })

    print(f"MTS done essay {essay_id}")
    time.sleep(5)

MTS done essay 66
MTS done essay 946
MTS done essay 837
MTS done essay 1710
MTS done essay 804
MTS done essay 1539
MTS done essay 1042
MTS done essay 241
MTS done essay 1211
MTS done essay 1485
MTS done essay 941
MTS done essay 1126
MTS done essay 1564
MTS done essay 1730
MTS done essay 1073
MTS done essay 922
MTS done essay 110
MTS done essay 1588
MTS done essay 558
MTS done essay 886
MTS done essay 355
MTS done essay 402
MTS done essay 1519
MTS done essay 1402
MTS done essay 325
MTS done essay 536
MTS done essay 1351
MTS done essay 699
MTS done essay 1691
MTS done essay 738


In [57]:
mts_results_df = pd.DataFrame(mts_results)
mts_results_df

,essay_id,human_score,human_category,raw_output,content_score,organization_score,language_score,average_score,predicted_score,predicted_category
0,66,9,High,Essay ID: 66\nContent Score: 4\nOrganization S...,4,4,4,4.000000,8,Medium
1,946,8,Medium,Essay ID: 946\nContent Score: 5\nOrganization ...,5,5,4,4.666667,9,High
2,837,8,Medium,Essay ID: 837\nContent Score: 5\nOrganization ...,5,4,4,4.333333,9,High
3,1710,9,High,Essay ID: 1710\nContent Score: 4\nOrganization...,4,4,3,3.666667,7,Medium
4,804,8,Medium,Essay ID: 804\nContent Score: 4\nOrganization ...,4,4,4,4.000000,8,Medium
5,1539,9,High,Essay ID: 1539\nContent Score: 5\nOrganization...,5,5,5,5.000000,10,High
6,1042,8,Medium,Essay ID: 1042\nContent Score: 5\nOrganization...,5,5,4,4.666667,9,High
7,241,9,High,Essay ID: 241\nContent Score: 4\nOrganization ...,4,3,2,3.000000,6,Medium
8,1211,9,High,Essay ID: 1211\nContent Score: 5\nOrganization...,5,5,4,4.666667,9,High
9,1485,9,High,Essay ID: 1485\nContent Score: 5\nOrganization...,5,5,4,4.666667,9,High


In [58]:
mts_clean_df = mts_results_df.dropna(subset=["predicted_score"]).copy()

mts_qwk = cohen_kappa_score(
    mts_clean_df["human_score"],
    mts_clean_df["predicted_score"],
    weights="quadratic",
)

mts_pcc, _ = pearsonr(
    mts_clean_df["human_score"],
    mts_clean_df["predicted_score"]
)

mts_mae = mean_absolute_error(
    mts_clean_df["human_score"],
    mts_clean_df["predicted_score"]
)

print("MTS QWK:", mts_qwk)
print("MTS PCC:", mts_pcc)
print("MTS MAE:", mts_mae)

MTS QWK: 0.5239294710327456
MTS PCC: 0.5564123357982286
MTS MAE: 1.2333333333333334


**ENHANCED PROMPT**

In [60]:
def get_response_enhanced(prompt, model="llama-3.3-70b-versatile", max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are an expert AES rater. "
                            "Follow the rubric and calibration exactly. "
                            "Return valid JSON only."
                        )
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0.05,
                max_tokens=250,
                top_p=0.88
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(5 * (attempt + 1))

    return None

In [61]:
##enhanced calibration examples
calibration_pool = (
    set1.groupby("domain1_score", group_keys=False)
    .apply(lambda x: x.sample(min(1, len(x)), random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

calibration = calibration_pool.sample(6, random_state=RANDOM_STATE).reset_index(drop=True)
calibration

/tmp/ipykernel_8246/2809319126.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(1, len(x)), random_state=RANDOM_STATE))


,essay_id,essay,domain1_score,score_category
0,1277,"Dear @CAPS1 @CAPS2, I am writing you this lett...",7,Medium
1,1592,The computers are cool. Do you now I werpsite ...,2,Low
2,1304,Would you like to discover new things and expl...,11,High
3,953,"Dear @CAPS1 of @LOCATION1, @CAPS2 the world ha...",12,High
4,19,I aegre waf the evansmant ov tnachnolage. The ...,4,Low
5,22,Dear local Newspaper @CAPS1 a take all your co...,3,Low


In [62]:
calibration_ids = calibration["essay_id"].tolist()

pool = set1[~set1["essay_id"].isin(calibration_ids)].copy().reset_index(drop=True)

demo_30 = pool.sample(30, random_state=RANDOM_STATE).reset_index(drop=True)

print("Calibration:", calibration.shape)
print("Demo 30:", demo_30.shape)

Calibration: (6, 4)
Demo 30: (30, 4)


In [63]:
rubric_text = """
Essay Prompt:
Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.

Rubric:
1 – Undeveloped response with minimal support.
2 – Underdeveloped response with weak reasoning.
3 – Minimally developed argument with limited support.
4 – Adequate argument with some supporting details.
5 – Developed argument with clear reasoning and organization.
6 – Strong persuasive response with well-developed ideas.

Scoring rule:
Two raters assign scores from 1–6.
Final score = sum of both scores.
Final score range: 2–12.
"""

In [64]:
def format_calibration_examples_with_id(calibration_df):
    examples = []
    for i, row in enumerate(calibration_df.itertuples(index=False), start=1):
        examples.append(f"""
Calibration Example {i}

Essay ID: {row.essay_id}

Essay:
{row.essay}

Human Score: {row.domain1_score}
Category: {row.score_category}
""")
    return "\n".join(examples)

calibration_text_with_id = format_calibration_examples_with_id(calibration)

In [65]:
def build_mts_prompt_enhanced(essay_id, essay_text, rubric_text, calibration_text_with_id):
    return f"""
<rubric>
{rubric_text}
Note: Final domain1_score = rater1(1-6) + rater2(1-6), so final scores range from 2 to 12.
</rubric>

<trait_guidance>
Content = relevance of ideas, argument development, supporting details
Organization = structure, coherence, logical flow, paragraphing
Language = grammar, vocabulary, clarity, sentence control
</trait_guidance>

<calibration>
{calibration_text_with_id}
Pattern:
- Low (2-5) = weak development and weak organization
- Medium (6-8) = adequate support and reasonable structure
- High (9-12) = strong reasoning, development, and structure
</calibration>

<task>
Essay ID: {essay_id}
Essay:
{essay_text}
</task>

<think>
1. Find the closest calibration example by overall writing quality.
2. Judge Content, Organization, and Language separately using the rubric.
3. Use the full trait range from 1 to 6 when justified.
4. Strong essays should receive 5s or 6s on multiple traits.
5. Return valid JSON only.
</think>

<output_format>
{{
  "essay_id": {essay_id},
  "content": X,
  "organization": X,
  "language": X,
  "reasoning": "2 short sentences"
}}
</output_format>
"""

In [66]:
##json parser
def parse_json_output(text):
    if not isinstance(text, str):
        return {}

    try:
        json_match = re.search(r'\{.*\}', text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except Exception:
        pass

    parsed = {}

    content_match = re.search(r'"?content"?\s*:\s*(\d+)', text, re.IGNORECASE)
    org_match = re.search(r'"?organization"?\s*:\s*(\d+)', text, re.IGNORECASE)
    lang_match = re.search(r'"?language"?\s*:\s*(\d+)', text, re.IGNORECASE)
    essay_id_match = re.search(r'"?essay_id"?\s*:\s*(\d+)', text, re.IGNORECASE)
    reasoning_match = re.search(r'"?reasoning"?\s*:\s*"([^"]+)"', text, re.IGNORECASE)

    if essay_id_match:
        parsed["essay_id"] = int(essay_id_match.group(1))
    if content_match:
        parsed["content"] = int(content_match.group(1))
    if org_match:
        parsed["organization"] = int(org_match.group(1))
    if lang_match:
        parsed["language"] = int(lang_match.group(1))
    if reasoning_match:
        parsed["reasoning"] = reasoning_match.group(1)

    return parsed

In [67]:
def clean_trait_score(value):
    if value is None:
        return None
    try:
        value = int(value)
        return max(1, min(6, value))
    except Exception:
        return None

In [68]:
def compute_mts_score(content, organization, language):
    if None in [content, organization, language]:
        return None, None, None

    avg = np.mean([content, organization, language])
    final_score = int(round(avg * 2))
    final_score = max(2, min(12, final_score))

    if final_score <= 5:
        category = "Low"
    elif final_score <= 8:
        category = "Medium"
    else:
        category = "High"

    return avg, final_score, category

In [70]:
mts_results_enhanced = []

for _, row in demo_30.iterrows():
    essay_id = row["essay_id"]
    essay_text = row["essay"]
    human_score = row["domain1_score"]
    human_category = row["score_category"]

    prompt = build_mts_prompt_enhanced(
        essay_id=essay_id,
        essay_text=essay_text,
        rubric_text=rubric_text,
        calibration_text_with_id=calibration_text_with_id
    )

    output = get_response_enhanced(prompt, model="llama-3.3-70b-versatile")

    if output is None:
        print(f"Skipping essay {essay_id}")
        mts_results_enhanced.append({
            "essay_id": essay_id,
            "human_score": human_score,
            "human_category": human_category,
            "raw_output": None,
            "content_score": None,
            "organization_score": None,
            "language_score": None,
            "average_score": None,
            "predicted_score": None,
            "predicted_category": None
        })
        continue

    parsed = parse_json_output(output)

    content_score = clean_trait_score(parsed.get("content"))
    organization_score = clean_trait_score(parsed.get("organization"))
    language_score = clean_trait_score(parsed.get("language"))

    average_score, predicted_score, predicted_category = compute_mts_score(
        content_score, organization_score, language_score
    )

    mts_results_enhanced.append({
        "essay_id": essay_id,
        "human_score": human_score,
        "human_category": human_category,
        "raw_output": output,
        "content_score": content_score,
        "organization_score": organization_score,
        "language_score": language_score,
        "average_score": average_score,
        "predicted_score": predicted_score,
        "predicted_category": predicted_category
    })

    print(f"Enhanced MTS done essay {essay_id}")
    time.sleep(2)

Enhanced MTS done essay 68
Enhanced MTS done essay 946
Enhanced MTS done essay 837
Enhanced MTS done essay 1710
Enhanced MTS done essay 804
Enhanced MTS done essay 1541
Enhanced MTS done essay 1042
Enhanced MTS done essay 242
Enhanced MTS done essay 1211
Enhanced MTS done essay 1487
Enhanced MTS done essay 941
Enhanced MTS done essay 1126
Enhanced MTS done essay 1566
Enhanced MTS done essay 1730
Enhanced MTS done essay 1073
Enhanced MTS done essay 922
Enhanced MTS done essay 112
Enhanced MTS done essay 1588
Enhanced MTS done essay 559
Enhanced MTS done essay 886
Enhanced MTS done essay 356
Enhanced MTS done essay 403
Enhanced MTS done essay 1521
Enhanced MTS done essay 1404
Enhanced MTS done essay 326
Enhanced MTS done essay 537
Enhanced MTS done essay 1353
Enhanced MTS done essay 699
Enhanced MTS done essay 1691
Enhanced MTS done essay 738


In [71]:
mts_results_enhanced_df = pd.DataFrame(mts_results_enhanced)
mts_results_enhanced_df

,essay_id,human_score,human_category,raw_output,content_score,organization_score,language_score,average_score,predicted_score,predicted_category
0,68,9,High,"{\n ""essay_id"": 68,\n ""content"": 4,\n ""orga...",4,4,4,4.000000,8,Medium
1,946,8,Medium,"{\n ""essay_id"": 946,\n ""content"": 4,\n ""org...",4,4,4,4.000000,8,Medium
2,837,8,Medium,"{\n ""essay_id"": 837,\n ""content"": 4,\n ""org...",4,4,4,4.000000,8,Medium
3,1710,9,High,"{\n ""essay_id"": 1710,\n ""content"": 4,\n ""or...",4,4,4,4.000000,8,Medium
4,804,8,Medium,"{\n ""essay_id"": 804,\n ""content"": 4,\n ""org...",4,4,4,4.000000,8,Medium
5,1541,8,Medium,"{\n ""essay_id"": 1541,\n ""content"": 5,\n ""or...",5,5,5,5.000000,10,High
6,1042,8,Medium,"{\n ""essay_id"": 1042,\n ""content"": 5,\n ""or...",5,5,4,4.666667,9,High
7,242,9,High,"{\n ""essay_id"": 242,\n ""content"": 4,\n ""org...",4,4,4,4.000000,8,Medium
8,1211,9,High,"{\n ""essay_id"": 1211,\n ""content"": 5,\n ""or...",5,5,4,4.666667,9,High
9,1487,8,Medium,"{\n ""essay_id"": 1487,\n ""content"": 4,\n ""or...",4,4,4,4.000000,8,Medium


In [78]:
mts_clean_enhanced_df = mts_results_enhanced_df.dropna(subset=["predicted_score"]).copy()

mts_qwk_enhanced = cohen_kappa_score(
    mts_clean_enhanced_df["human_score"],
    mts_clean_enhanced_df["predicted_score"],
    weights="quadratic",

)

mts_pcc_enhanced, _ = pearsonr(
    mts_clean_enhanced_df["human_score"],
    mts_clean_enhanced_df["predicted_score"]
)

mts_mae_enhanced = mean_absolute_error(
    mts_clean_enhanced_df["human_score"],
    mts_clean_enhanced_df["predicted_score"]
)

print("Enhanced MTS QWK:", mts_qwk_enhanced)
print("Enhanced MTS PCC:", mts_pcc_enhanced)
print("Enhanced MTS MAE:", mts_mae_enhanced)
print("Valid predictions:", len(mts_clean_enhanced_df), "/", len(mts_results_enhanced_df))

Enhanced MTS QWK: 0.5922330097087378
Enhanced MTS PCC: 0.7091776838860101
Enhanced MTS MAE: 0.9333333333333333
Valid predictions: 30 / 30


In [75]:
mts_results_enhanced_df["abs_error"] = (
    mts_results_enhanced_df["human_score"] - mts_results_enhanced_df["predicted_score"]
).abs()

mts_results_enhanced_df.sort_values("abs_error", ascending=False)[
    ["essay_id", "human_score", "predicted_score", "content_score", "organization_score", "language_score", "abs_error"]
].head(10)

,essay_id,human_score,predicted_score,content_score,organization_score,language_score,abs_error
5,1541,8,10,5,5,5,2
11,1126,10,8,4,4,4,2
10,941,12,10,5,5,5,2
29,738,6,8,4,4,4,2
27,699,12,10,5,5,5,2
19,886,11,9,5,4,4,2
28,1691,11,9,5,5,4,2
13,1730,10,9,5,5,4,1
7,242,9,8,4,4,4,1
21,403,8,7,4,4,3,1
